# SEC plot workflow: raw data → normalized Excel → SVG

This notebook documents the full pipeline:

0. **Raw → normalized Excel** — Build the "Data" sheet from raw instrument CSV exports (one CSV per condition). Combines four runs, normalizes each trace to [0, 1], interpolates onto a common volume grid, and writes an Excel file that downstream steps expect.
1. **Batch SVG generation** — Python script reading those Excel files from a `data/` folder and writing SVGs to `output/`.
2. **Web GUI** — Flask backend plus HTML front end for uploading CSV, editing legend text, and generating SVG.

Organization and documentation use relative paths only, clear structure, and no spaces in directory names.

**Data:** Raw CSVs live in a folder per protein (e.g. `BSA_raw_data/`). The normalized Excel can be written to `data/` for the batch SVG step.


---

## 0. Raw instrument CSVs → normalized Excel (Data sheet)

The second tab of files like `bsa_curves_normalized.xlsx` (the **Data** sheet) is built from the **raw CSV exports** in a folder such as `BSA_raw_data/`. It is **not** from a single file: four CSVs (one per condition) are combined.

**Relationship:**

- Each raw CSV is one chromatograph run: tab-separated, UTF-16, with 3 header rows. Volume (ml) and UV (mAU) are in **columns 0 and 1** (first UV channel).
- The folder contains one CSV per condition (e.g. BSA with TCEP, BSA standards, BSA EDTA, BSA TCEP+EDTA). The `BSA_curves/` subfolder holds intermediate workbooks; the final Data sheet can be reproduced by reading the **same four CSVs** directly.
- **Steps:** (1) Read each CSV → (vol, UV). (2) Normalize UV to [0, 1] per run: `(UV - min) / (max - min)`. (3) Interpolate each normalized trace onto a **common volume grid** (e.g. 299 points from 0.1 to 25 mL). (4) Assemble one table: four (Volume, signal) column pairs plus Pk_x_0..3, Pk_y_0..3 (peak volume and normalized peak = 1 for each series). (5) Write one Excel sheet (Data).

The module `sec_raw_to_normalized.py` implements this. Below we run it for BSA: point at the raw folder and a filename→condition mapping, then write the normalized Excel.


In [ ]:
from pathlib import Path
import sys

%load_ext autoreload
%autoreload 2

REPO_ROOT = Path.cwd().parent

SEC_ANALYSIS = REPO_ROOT / "SEC_analysis"

if SEC_ANALYSIS not in sys.path:
    sys.path.insert(0, str(SEC_ANALYSIS))

In [ ]:
from sec_raw_to_normalized import run_all_proteins


SEC_DATA_DIR = SEC_ANALYSIS / "data"
DATA_DIR = SEC_ANALYSIS / "normalized"
OUTPUT_DIR = Path("plots") / "SEC"


DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run_all_proteins(SEC_DATA_DIR, DATA_DIR, volume_min=0.1, volume_max=25.0)


**Proteins supported:** BSA, DNAse I, Cytochrome c, Hemoglobin. The mapping is:

| Raw folder                      | Normalized Excel file             |
| ------------------------------- | --------------------------------- |
| `SEC data/BSA_raw_data/`        | `bsa_curves_normalized.xlsx`      |
| `SEC data/DNAse_raw_data/`      | `DNAse_curves_normalized.xlsx`    |
| `SEC data/CytC_raw_data/`       | `Cyt_C_curves_normalized.xlsx`    |
| `SEC data/Hemoglobin_raw_data/` | `Hemo_curves_normalized (2).xlsx` |

All four use the same pipeline; `sec_raw_to_normalized.SEC_PROTEIN_CONFIG` defines the CSV→condition mapping per protein. If a raw folder is missing, that protein is skipped. The Chart tab is not recreated; the Data sheet is enough for plotting and the web GUI.


---

## 1. Batch SVG generation (local script)

The script `plot_sec_to_svg.py`:

- Reads all `.xlsx` files from a single input directory (here: `data/`).
- For each file, loads the **Data** sheet and detects column pairs (Volume, signal), stopping at `Pk_*` columns.
- Builds series with peak volumes, applies Arcadia colors and legend order (No Treatment, TCEP, EDTA, TCEP + EDTA), and renders one SVG per file.
- Writes SVGs to an output directory (here: `output/`).

Below we run the same logic from the notebook using relative paths so the repo is self-contained and path-agnostic.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from plot_sec_to_svg import get_series_pairs, render_sec_svg

xlsx_files = sorted(DATA_DIR.glob("*.xlsx"))
xlsx_files

**Single-file example:** Load one Excel file from `data/`, get series pairs from the Data sheet, render to SVG, and save to `output/`.


In [ ]:
# path = xlsx_files[0]
# df = pd.read_excel(path, sheet_name="Data", header=0)
# pairs = get_series_pairs(df)
# title = path.stem.replace("_", " ").replace(" (2)", "").replace(" (27)", "")
# svg = render_sec_svg(pairs, title)
# out_path = OUTPUT_DIR / (path.stem + ".svg")
# out_path.write_text(svg, encoding="utf-8")
# print(f"Saved {out_path.name}")


**Batch run:** Loop over all `.xlsx` files in `data/` and write one SVG per file (same behavior as running `python plot_sec_to_svg.py` when the script is configured to use `data/` and `output/`).


In [ ]:
for excel_path in xlsx_files:
    xl = pd.ExcelFile(excel_path)
    if "Data" not in xl.sheet_names:
        print(f"  Skip {excel_path.name}: no 'Data' sheet")
        continue
    df = pd.read_excel(excel_path, sheet_name="Data", header=0)
    pairs = get_series_pairs(df)
    if not pairs:
        print(f"  Skip {excel_path.name}: no series found")
        continue
    title = excel_path.stem.replace("_", " ").replace(" (2)", "").replace(" (27)", "")
    render_sec_svg(pairs, title)

print("Done.")
